In [46]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import sklearn as skl
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics 

based on TSNE/PCA results, the 0 risk class seems to be the most separated, while the 1, 2, and 3 risk classes are more mixed. 
The 3 risk class is more separated, but still fairly mixed with class 1 and 2.  

Based on this, we decided to separate the target class into two different splits for logistic regression. The first option is using   
class 0 ("Low" risk) as logreg class 0 and all other classes (1-3, "moderate" to "severe" risk) as logreg class 1.  
The second option is splitting the classes in half, so that "low" and "moderate" (0-1) are logreg class 0 and "high" and "severe" (2-3)
as logreg class 1. 

In [47]:
def readData(datafile):
    # get the current directory: directory name for the abs path of the curr file
    dirpath = os.getcwd()
    abspath = dirpath + "\\" + datafile

    # read data into a pandas dataframe. data file is comma separated so use read_csv
    df = pd.read_csv(abspath)
     
    return df

In [48]:
def runLogReg(features, target):
    # TEST is completely unseen data (20%), TRAIN is split into TRAIN (0.8 of 0.8 = 64%) and VAL (16%)
    # x is the feature data and y is the target
    xtrain, xtest, ytrain, ytest = train_test_split(features, target, test_size=0.2, shuffle=True)
    #xtrain, xval, ytrain, yval = train_test_split(xtrain, ytrain, test_size=0.2)

    # only scale the features!! the target has to be kept as discrete vals and scaling would make continuous
    scaler = skl.preprocessing.StandardScaler()
    xtrain = scaler.fit_transform(xtrain)
    xtest = scaler.fit_transform(xtest)

    # create the model and train
    # C is inverse of regularization, so large C means small regularization
    logReg = LogisticRegression(C=10, max_iter=1000)
    logReg.fit(xtrain, ytrain)
    ypred = logReg.predict(xtest)
    yprob = logReg.predict_proba(xtest)[:,1]

    # metrics
    fpr, tpr, _ = metrics.roc_curve(ytest, yprob)
    auc = metrics.auc(fpr, tpr)
    f1 = metrics.f1_score(ytest, ypred)
    accuracy = metrics.accuracy_score(ytest, ypred)

    return auc, f1, accuracy

In [49]:
# read in data
df = readData("GamingandMentalHealth_final.csv")

# map to split 1: "low"/0 risk is still 0, all other values are now 1
map1 = {0:0, 1:1, 2:1, 3:1}
data1 = df.copy()
data1.iloc[:,-1] = data1.iloc[:, -1].map(map1)

# set up for test train split
data1 = data1.to_numpy()
features = data1[:,0:-1]
target = data1[:,-1].astype(int)

# run the logisitic regression model
auc_1, f1_1, accuracy_1 = runLogReg(features, target)

# print metrics 
print(f"Logistic Regression model with target split 1 ('Low'=0, 'Moderate' and above=1)")
print(f"AUC: {auc_1:.6f} F1: {f1_1:.6f} Accuracy: {accuracy_1:.6f}\n")


# map to split 2: low/moderate=0, high/severe=1
map2 = {0:0, 1:0, 2:1, 3:1}
data2 = df.copy()
data2.iloc[:,-1] = data2.iloc[:, -1].map(map2)

# set up for test train split
data2 = data2.to_numpy()
features = data2[:,0:-1]
target = data2[:,-1].astype(int)

# run the logisitic regression model and print metrics
auc_2, f1_2, accuracy_2 = runLogReg(features, target)
print(f"Logistic Regression model with target split 2 ('Low'/'Moderate'=0, 'High'/'Severe'=1)")
print(f"AUC: {auc_2:.6f} F1: {f1_2:.6f} Accuracy: {accuracy_2:.6f}")



Logistic Regression model with target split 1 ('Low'=0, 'Moderate' and above=1)
AUC: 1.000000 F1: 0.994764 Accuracy: 0.995000

Logistic Regression model with target split 2 ('Low'/'Moderate'=0, 'High'/'Severe'=1)
AUC: 1.000000 F1: 0.970874 Accuracy: 0.985000
